# Greedy Algorithms: Principles, Examples, and Limitations

**Algorithms and Data Structures**  
**Riga Technical University**  

Lector: Valdis Saulespurens  - valdis.saulespurens@rtu.lv

Spring 2026


![Burns](https://upload.wikimedia.org/wikipedia/en/5/56/Mr_Burns.png) | ![Gordon](https://upload.wikimedia.org/wikipedia/en/4/40/Gordon_Gekko.jpg)

From: https://en.wikipedia.org/wiki/Wall_Street_(1987_film)

The subfamily of *Greedy Algorithms* is one of the main paradigms of algorithmic problem solving next to *Dynamic Programming* and *Divide & Conquer Algorithms*. The main goal behind greedy algorithms is to implement an efficient procedure for often computationally more complex, often infeasible brute-force methods such as exhaustive search algorithms. 

The main outline of a greedy algorithms consists of 3 steps:

- make a greedy choice
- reduce the problem to a subproblem (a smaller problem of the similar type as the original one - what happens if we do not reduce?)
- repeat

So, greedy algorithms are essentially a problem solving heuristic, an iterative process of tackling a problem in multiple stages while making an locally optimal choice at each stage. In practice, and depending on the problem task, making this series of locally optimal ("greedy") choices must not necessarily lead to a globally optimal solution.

## Learning Goals

After this lecture you should be able to:

- explain what a greedy algorithm is
- distinguish **local optimal choices** from **global optimal solutions**
- implement several classic greedy algorithms in Python
- recognize situations where greedy algorithms succeed
- recognize situations where greedy algorithms fail
- explain the basic correctness intuition for several greedy algorithms

This notebook is designed for **Python 3.12+** and should run both:

- locally in **Visual Studio Code** using a virtual environment
- in **Google Colab**


## What Is a Greedy Algorithm?

A **greedy algorithm** builds a solution step by step.

At each step it chooses the option that looks best **right now**, without reconsidering previous decisions.

Typical characteristics:

- decisions are **locally optimal**
- the algorithm usually **does not backtrack**
- the algorithm does **not explore all possible solutions**

Greedy algorithms are attractive because they are often:

- simple
- fast
- easy to implement

However, greedy algorithms are **not always correct**.

A central question in algorithm design is:

> When does a greedy strategy produce a globally optimal solution?


![Greedy Mountain](https://upload.wikimedia.org/wikipedia/commons/8/8c/Greedy-search-path-example.gif)

## Intuition: Local Choices vs Global Optimality

Greedy algorithms rely on the hope that repeated **good local choices** lead to a **globally optimal solution**.

Sometimes that hope is justified.

Sometimes it is not.

Throughout this notebook we will repeatedly ask:

1. What greedy rule might we try?
2. Does it work?
3. If it works, why?
4. If it fails, where does the reasoning break?


In [1]:
from __future__ import annotations

from typing import Iterable, Sequence

def print_rule(title: str) -> None:
    print("=" * len(title))
    print(title)
    print("=" * len(title))


## Example 1 — Coin Change Problem

**Problem.**  
Given coin denominations and a target value, find the **minimum number of coins** needed to represent the value.

Example:

- Coins: `[1, 5, 10, 25]`
- Target: `63`

A natural greedy strategy is:

> Always take the largest coin that does not exceed the remaining amount.


### Greedy Coin Change Algorithm


In [ ]:
def greedy_coin_change(amount: int, coins: Sequence[int], debug: bool = False) -> list[int]:
    """Return a greedy coin decomposition for the given amount.

    Parameters
    ----------
    amount:
        Target amount to represent. Must be non-negative.
    coins:
        Available denominations. They do not need to be sorted.
    debug:
        If True, print the greedy choices step by step.

    Returns
    -------
    list[int]
        A list of selected coin denominations in greedy order.

    Raises
    ------
    ValueError
        If amount is negative, coin values are invalid, or exact change is impossible.
    """
    if amount < 0:
        raise ValueError("amount must be non-negative")
    if not coins:
        raise ValueError("coins must not be empty")
    if any(c <= 0 for c in coins):
        raise ValueError("all coin denominations must be positive")

    remaining = amount
    result: list[int] = []
    sorted_coins = sorted(set(coins), reverse=True)

    if debug:
        print_rule("Greedy coin change")
        print(f"Amount: {amount}")
        print(f"Coins:  {sorted_coins}")

    for coin in sorted_coins:
        while remaining >= coin:
            result.append(coin)
            remaining -= coin
            if debug:
                print(f"Take {coin:>2}; remaining = {remaining}")

    if remaining != 0:
        raise ValueError(
            "exact change is impossible with the given coin system"
        )

    if debug:
        print(f"Solution: {result}")
        print(f"Number of coins: {len(result)}")

    return result


### Examples


In [ ]:
print_rule("Example 1A: standard coin system")
solution = greedy_coin_change(63, [1, 5, 10, 25], debug=True)
print()

print_rule("Example 1B: euro-like system")
print(greedy_coin_change(88, [1, 2, 5, 10, 20, 50]))


### When Greedy Works

Many familiar currency systems are designed so that greedy coin change works correctly in practice.

That does **not** mean greedy is automatically correct for **all** coin systems.


### When Greedy Fails

Consider the coin system `[1, 3, 4]` and target `6`.

Greedy chooses:

- `4`, leaving `2`
- then `1`, leaving `1`
- then `1`

So greedy returns `4 + 1 + 1`, which uses **3 coins**.

But the optimal answer is:

- `3 + 3`, which uses **2 coins**

This shows a common greedy pattern:

> a locally best first move can block a globally better solution


In [ ]:
print_rule("Example 1C: counterexample where greedy fails")
greedy = greedy_coin_change(6, [1, 3, 4], debug=True)
print(f"Greedy result: {greedy} -> {len(greedy)} coins")
print("Optimal result: [3, 3] -> 2 coins")


### Takeaway

Greedy coin change is fast and intuitive, but correctness depends on the structure of the coin system, not on intuition alone.


## Example 2 — Activity Selection Problem

**Problem.**  
We are given a set of activities, each with a start time and a finish time.

**Goal.**  
Select the **maximum number of non-overlapping activities**.


### Representation

Each activity will be represented as a tuple:

`(start_time, finish_time)`


### Plausible Greedy Strategies

Before using the correct strategy, it is worth noting that several natural-looking rules can fail:

1. choose the activity with the earliest start time
2. choose the activity with the shortest duration
3. choose the activity with the fewest apparent conflicts

These rules sound reasonable, but they do **not** guarantee an optimal solution.


### A Greedy Strategy That Works

The correct greedy rule is:

> Always choose the compatible activity with the **earliest finishing time**.

The intuition is that finishing earlier leaves as much room as possible for future activities.


### Activity Selection Algorithm


In [ ]:
Activity = tuple[int, int]

def activity_selection(activities: Sequence[Activity], debug: bool = False) -> list[Activity]:
    """Select a maximum-size set of pairwise non-overlapping activities.

    Greedy rule: repeatedly choose the next compatible activity that finishes earliest.
    """
    if any(start > finish for start, finish in activities):
        raise ValueError("each activity must satisfy start <= finish")

    sorted_activities = sorted(activities, key=lambda act: (act[1], act[0]))
    selected: list[Activity] = []
    current_finish = -10**18

    if debug:
        print_rule("Activity selection")
        print("Activities sorted by finish time:")
        for act in sorted_activities:
            print(f"  {act}")

    for activity in sorted_activities:
        start, finish = activity
        if start >= current_finish:
            selected.append(activity)
            current_finish = finish
            if debug:
                print(f"Take {activity}; current_finish = {current_finish}")
        elif debug:
            print(f"Skip {activity}; overlaps with selected schedule")

    if debug:
        print(f"Selected activities: {selected}")
        print(f"Maximum number found by greedy: {len(selected)}")

    return selected


### Example


In [ ]:
activities = [
    (1, 4),
    (3, 5),
    (0, 6),
    (5, 7),
    (3, 9),
    (5, 9),
    (6, 10),
    (8, 11),
    (8, 12),
    (2, 14),
    (12, 16),
]

selected = activity_selection(activities, debug=True)
print()
print(f"Final schedule: {selected}")


### Why Does This Work?

A standard correctness intuition uses an **exchange argument**.

Suppose an optimal solution starts with some activity `A`, but the greedy algorithm chooses a different compatible activity `G` that finishes earlier.

Because `G` finishes no later than `A`, replacing `A` by `G` still leaves room for all later activities that were compatible with `A`.

So there exists an optimal solution that begins with the greedy choice.

Repeating this reasoning step by step justifies the greedy algorithm.


### A Wrong Greedy Idea: Earliest Start Time

Choosing the earliest starting activity can fail, because an early-starting activity may last too long and block many shorter later activities.


In [ ]:
wrong_example = [
    (0, 100),  # starts earliest, but blocks almost everything
    (1, 2),
    (2, 3),
    (3, 4),
    (4, 5),
]

print_rule("Wrong heuristic: earliest start time")
print("If we choose (0, 100), we get only one activity.")
print("Greedy by earliest finish time gives:")
print(activity_selection(wrong_example))


## Example 3 — Knapsack Problem

We are given items with:

- a **value**
- a **weight**

Goal:

- maximize total value
- without exceeding a weight capacity


### Two Versions of the Problem

1. **0-1 Knapsack**  
   Each item is either taken whole or not taken at all.

2. **Fractional Knapsack**  
   We may take fractions of items.

These two problems look similar, but their greedy behavior is very different.


### Greedy Strategy

A natural greedy idea is to sort items by **value / weight ratio** and take the best ratio first.


In [ ]:
Item = tuple[str, float, float]  # (name, value, weight)

def fractional_knapsack(items: Sequence[Item], capacity: float, debug: bool = False) -> tuple[float, list[tuple[str, float]]]:
    """Solve fractional knapsack greedily by value/weight ratio.

    Returns
    -------
    tuple[float, list[tuple[str, float]]]
        Total value and a list of (item_name, fraction_taken).
    """
    if capacity < 0:
        raise ValueError("capacity must be non-negative")
    if any(weight <= 0 for _, _, weight in items):
        raise ValueError("all item weights must be positive")

    remaining = capacity
    total_value = 0.0
    chosen: list[tuple[str, float]] = []

    sorted_items = sorted(items, key=lambda item: item[1] / item[2], reverse=True)

    if debug:
        print_rule("Fractional knapsack")
        print(f"Capacity: {capacity}")
        print("Items sorted by value/weight:")
        for name, value, weight in sorted_items:
            print(f"  {name}: value={value}, weight={weight}, ratio={value/weight:.2f}")

    for name, value, weight in sorted_items:
        if remaining == 0:
            break

        take_weight = min(weight, remaining)
        fraction = take_weight / weight
        total_value += value * fraction
        chosen.append((name, fraction))
        remaining -= take_weight

        if debug:
            print(
                f"Take {fraction:.2f} of {name}; "
                f"added value = {value * fraction:.2f}; remaining capacity = {remaining:.2f}"
            )

    if debug:
        print(f"Total value = {total_value:.2f}")
        print(f"Chosen fractions = {chosen}")

    return total_value, chosen


### Example: Fractional Knapsack


In [ ]:
items: list[Item] = [
    ("gold", 60, 10),
    ("silver", 100, 20),
    ("bronze", 120, 30),
]

value, chosen = fractional_knapsack(items, capacity=50, debug=True)
print()
print(f"Optimal fractional value: {value:.2f}")
print(f"Chosen fractions: {chosen}")


### Why Greedy Works Here

In the fractional version, if the best remaining item does not fully fit, we can still take the exact fraction that fits.

That makes the ratio-based greedy choice safe.


### Why Greedy Fails for 0-1 Knapsack

In 0-1 knapsack we cannot split items.

A high-ratio item may look best locally, but taking it can block a better combination of whole items later.

So the same greedy rule is **not** guaranteed to be correct.


In [ ]:
print_rule("0-1 knapsack counterexample")
items_01 = [
    ("A", 60, 10),   # ratio 6.0
    ("B", 100, 20),  # ratio 5.0
    ("C", 120, 30),  # ratio 4.0
]
capacity_01 = 50

print("Greedy by ratio for 0-1 knapsack would choose A and B:")
print("  value = 160, weight = 30")
print("Then C no longer fits.")
print("But the optimal 0-1 solution is B and C:")
print("  value = 220, weight = 50")


## Example 4 — Minimum Points to Cover Intervals

**Problem.**  
Given intervals on a number line, choose the **minimum number of points** such that every interval contains at least one chosen point.


### Greedy Idea

1. Sort intervals by their **right endpoint**
2. Choose the right endpoint of the first interval
3. Remove all intervals covered by that point
4. Repeat


In [ ]:
Interval = tuple[int, int]

def minimum_cover_points(intervals: Sequence[Interval], debug: bool = False) -> list[int]:
    """Return a minimum-size set of points covering all intervals."""
    if any(left > right for left, right in intervals):
        raise ValueError("each interval must satisfy left <= right")

    if not intervals:
        return []

    sorted_intervals = sorted(intervals, key=lambda iv: (iv[1], iv[0]))
    points: list[int] = []
    current_point: int | None = None

    if debug:
        print_rule("Minimum points to cover intervals")
        print("Intervals sorted by right endpoint:")
        for interval in sorted_intervals:
            print(f"  {interval}")

    for left, right in sorted_intervals:
        if current_point is None or current_point < left:
            current_point = right
            points.append(current_point)
            if debug:
                print(f"Choose point {current_point} to cover interval ({left}, {right})")
        elif debug:
            print(f"Interval ({left}, {right}) already covered by point {current_point}")

    if debug:
        print(f"Chosen points: {points}")

    return points


### Example


In [ ]:
intervals = [(1, 3), (2, 5), (3, 6), (7, 9), (8, 10)]
points = minimum_cover_points(intervals, debug=True)
print()
print(f"Minimum cover points: {points}")


### Why This Works

When we pick the right endpoint of the earliest-ending uncovered interval, we cover that interval while preserving as much freedom as possible for later intervals.

Any point further right would miss the current interval.  
Any point further left is never better.


## Example 5 — Maximum Number of Distinct Summands

**Problem.**  
Given a positive integer `n`, represent it as the **maximum number of distinct positive integers**.

Example:

`8 = 1 + 2 + 5`


### Greedy Idea

Always take the **smallest unused positive integer** that still allows the remaining sum to be completed with a larger distinct number later.


In [ ]:
def max_distinct_summands(n: int, debug: bool = False) -> list[int]:
    """Represent n as a maximum-size sum of distinct positive integers."""
    if n <= 0:
        raise ValueError("n must be positive")

    summands: list[int] = []
    next_value = 1
    remaining = n

    if debug:
        print_rule("Maximum number of distinct summands")
        print(f"n = {n}")

    while remaining > 0:
        # If we take next_value now, then the leftover should either be 0
        # or strictly larger than next_value so that distinctness is preserved.
        if remaining - next_value > next_value:
            summands.append(next_value)
            remaining -= next_value
            if debug:
                print(f"Take {next_value}; remaining = {remaining}")
            next_value += 1
        else:
            summands.append(remaining)
            if debug:
                print(f"Take final summand {remaining}; remaining = 0")
            remaining = 0

    if debug:
        print(f"Summands: {summands}")
        print(f"Count: {len(summands)}")

    return summands


### Example


In [ ]:
summands = max_distinct_summands(8, debug=True)
print()
print(f"8 = {' + '.join(map(str, summands))}")


### Why This Works

Choosing the smallest available summand leaves as much room as possible for future distinct summands.

At the end, any leftover amount can be absorbed into the final summand without breaking distinctness.


## When Do Greedy Algorithms Work?

Two recurring ideas appear again and again.

### 1. Greedy-Choice Property

A globally optimal solution can be constructed by making a locally optimal choice first.

### 2. Optimal Substructure

After the first correct choice, the remaining problem is itself an optimal subproblem of the same general type.

Common proof styles include:

- **exchange arguments**
- **stays-ahead arguments**
- problem-specific structural reasoning


## When Do Greedy Algorithms Fail?

Greedy algorithms may fail when:

- an early choice blocks better later combinations
- local optimality conflicts with global optimality
- the problem does not have the greedy-choice property

Examples from this notebook:

- coin change for some coin systems
- 0-1 knapsack
- incorrect greedy rules for activity selection


## Local Optima and Hill Climbing

Greedy thinking is closely related to the idea of **hill climbing**.

Hill climbing repeatedly moves to a better neighboring solution.

This can work well, but it can also get stuck at a **local optimum** rather than reaching the **global optimum**.

This is one reason algorithm design requires proof, not just intuition.


In [ ]:
def hill_climb_demo(values: Sequence[int]) -> int:
    """A tiny 1D hill-climbing demonstration.

    Starting from index 0, move right while the next value is strictly larger.
    Return the index where the process stops.
    """
    if not values:
        raise ValueError("values must not be empty")

    i = 0
    while i + 1 < len(values) and values[i + 1] > values[i]:
        i += 1
    return i

landscape = [1, 3, 7, 6, 5, 10, 9]
stop_index = hill_climb_demo(landscape)

print_rule("Hill-climbing toy example")
print(f"Landscape: {landscape}")
print(f"Stopped at index {stop_index} with value {landscape[stop_index]}")
print("This is a local optimum relative to the chosen move rule, not the global maximum 10.")


## Summary

Greedy algorithms are powerful because they are often:

- simple
- efficient
- elegant

But greedy algorithms are **not automatically correct**.

The main lesson is:

> A greedy algorithm is only as good as the structural reason that justifies its local choice.

For each new problem, we should ask:

- what is the greedy rule?
- why should it work?
- can we prove it?
- can we find a counterexample?


## Optional Topics for Further Study

These topics are natural extensions but are omitted from the main notebook flow:

- Huffman coding - cool string compression algorithm based on greedy tree construction
- set cover approximation
- greedy graph coloring - fascinating topic with connections to graph theory and combinatorics
- minimum spanning trees - we cover this later in the course, so we will not cover it here
- Dijkstra's algorithm - we cover this later in the course, so we will not cover it here


## Exercises

1. Construct a coin system where greedy coin change fails for some target value.
2. Modify `activity_selection` so that it also reports the number of rejected activities.
3. Write a brute-force checker for small activity-selection instances and compare it with the greedy answer.
4. Create several random fractional knapsack instances and verify the greedy solution manually on small cases.
5. Prove correctness of the interval-covering algorithm using your own words.
6. For which values of `n` does `max_distinct_summands(n)` return exactly 3 summands?


## End of Notebook

This notebook is intended as a clean teaching baseline for live coding, classroom explanation, and later extension.

### License

MIT License (c) 2026 Valdis Saulespurens.
Feel free to use, modify, and share this material for educational purposes.
